# Cs 8P-to-52D ARC / rydcalc laser-power comparison

Goal: repeat the ARC-vs-rydcalc validation workflow for the cesium transition

```text
|8P1/2, mj=1/2> -> |52D3/2, mj=1/2>
```

Because `Delta mj = 0`, the selected electric-dipole polarization is `pi`, i.e. `q = 0`.

This notebook computes the effective dipole matrix element and Gaussian-beam optical power with ARC and with the local `rydcalc` submodule, then compares the numerical result with spectroscopy/Rydberg-atom literature.

Working beam/Rabi assumptions are intentionally the same as the previous Cs notebook unless otherwise stated:

- circular Gaussian beam at the atom, `w_x = w_y = 10 um`
- target `Omega/(2 pi) = 1, 5, 10 MHz`
- no optical loss, imperfect polarization, clipping, or alignment margin included


## Formula conventions

At the center of a Gaussian beam,

```text
Omega = |d_eff| E0 / hbar
I0 = 2 P / (pi w_x w_y)
I0 = c epsilon0 E0^2 / 2
```

Therefore

```text
P = pi w_x w_y c epsilon0 hbar^2 Omega^2 / (4 |d_eff|^2)
```

ARC and rydcalc both return electric-dipole matrix elements in atomic units `e a0`. The SI conversion is

```text
d_SI = d_au * e * a0
```

Only `abs(d_eff)` enters the power. The sign of the matrix element is a phase convention.


In [1]:
from __future__ import annotations

import inspect
import math
import os
import shutil
import sys
from pathlib import Path

import numpy as np
from scipy import constants as cs

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent

mpl_cache = Path("/tmp/matplotlib-arc-rydcalc-cs")
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

# NumPy 2 compatibility for the current local rydcalc submodule.
if not hasattr(np, "product"):
    np.product = np.prod
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

if str(root / "rydcalc") not in sys.path:
    sys.path.insert(0, str(root / "rydcalc"))

import arc
from arc import Caesium
import arc.alkali_atom_data as arc_atom_data
import rydcalc


def prepare_arc_data_cache() -> Path:
    source_candidates = [
        root / "ARC-Alkali-Rydberg-Calculator" / "arc" / "data",
        Path(inspect.getfile(arc)).resolve().parent / "data",
    ]
    source = next(path for path in source_candidates if path.exists())
    target = Path("/tmp/arc-data-cs-8p-52d")
    version = source / "version.txt"
    target_version = target / "version.txt"
    if not target.exists() or (version.exists() and (not target_version.exists() or target_version.read_text() != version.read_text())):
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(source, target)
    return target

arc_data = prepare_arc_data_cache()
Caesium.dataFolder = str(arc_data)
arc_atom_data.Caesium.dataFolder = str(arc_data)

print(f"project root: {root}")
print(f"ARC version: {getattr(arc, '__version__', 'unknown')}")
print(f"ARC data cache: {arc_data}")
print(f"rydcalc module: {Path(rydcalc.__file__).resolve()}")
print(f"NumPy compatibility aliases: product={hasattr(np, 'product')}, trapz={hasattr(np, 'trapz')}")

project root: /home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization
ARC version: 3.10.2
ARC data cache: /tmp/arc-data-cs-8p-52d
rydcalc module: /home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization/rydcalc/rydcalc/__init__.py
NumPy compatibility aliases: product=True, trapz=True


## Important package calls

ARC calls used here:

- `Caesium()` constructs the Cs model.
- `getDipoleMatrixElement(n1, l1, j1, mj1, n2, l2, j2, mj2, q)` returns the `mj`-resolved E1 matrix element in `e a0`.
- `getRadialMatrixElement(...)` and `getReducedMatrixElementJ(...)` expose the radial and reduced-J factors.
- `getTransitionWavelength(...)` returns the transition wavelength in vacuum meters.
- `getEnergy(...)` returns binding energy relative to ionization in eV in this ARC convention; adding the Cs ionization energy gives level energy above the ground state.

rydcalc calls used here:

- `rydcalc.Cesium133(use_db=False)` constructs the local Cs model without relying on a writable dipole database.
- `atom.get_state((n, l, j, m))` returns a state object.
- `state.energy_Hz` is the zero-field energy relative to the ionization limit, in Hz.
- `state.get_multipole_me(other, k=1, qIn=[0])` returns the `pi` E1 matrix element in `e a0`.


In [2]:
transition = {
    "n1": 8,
    "l1": 1,
    "j1": 0.5,
    "mj1": 0.5,
    "n2": 52,
    "l2": 2,
    "j2": 1.5,
    "mj2": 0.5,
    "q": 0,
}

w_x = 10e-6
w_y = 10e-6
target_rabi_hz = [1e6, 5e6, 10e6]
au_dipole_si = cs.e * cs.physical_constants["Bohr radius"][0]
cs_ionization_cm = 31406.46769
cm_per_ev = 8065.544005


def gaussian_power_for_rabi(rabi_hz: float, d_au: float, w_x: float, w_y: float) -> dict[str, float]:
    d_si = abs(d_au) * au_dipole_si
    omega = 2.0 * math.pi * rabi_hz
    power_w = math.pi * w_x * w_y * cs.c * cs.epsilon_0 * cs.hbar**2 * omega**2 / (4.0 * d_si**2)
    intensity_w_m2 = 2.0 * power_w / (math.pi * w_x * w_y)
    field_v_m = math.sqrt(2.0 * intensity_w_m2 / (cs.c * cs.epsilon_0))
    return {
        "rabi_hz": rabi_hz,
        "power_w": power_w,
        "intensity_w_m2": intensity_w_m2,
        "field_v_m": field_v_m,
    }


def print_power_block(label: str, d_au: float, wavelength_m: float, extra: dict[str, str] | None = None) -> None:
    d_si = d_au * au_dipole_si
    print(label)
    print(f"  lambda = {1e9 * wavelength_m:.6f} nm")
    print(f"  d_eff = {d_au:+.12e} e a0 = {d_si:+.12e} C m")
    if extra:
        for key, value in extra.items():
            print(f"  {key} = {value}")
    print(f"  Gaussian waist: wx = wy = {1e6 * w_x:.1f} um")
    for result in [gaussian_power_for_rabi(rabi, d_au, w_x, w_y) for rabi in target_rabi_hz]:
        print(
            f"  Omega/2pi = {result['rabi_hz'] / 1e6:>4.1f} MHz -> "
            f"P = {result['power_w']:.9e} W = {1e3 * result['power_w']:.6f} mW, "
            f"I0 = {result['intensity_w_m2']:.6e} W/m^2, "
            f"E0 = {result['field_v_m']:.6e} V/m"
        )

In [3]:
arc_cs = Caesium()

arc_dme = arc_cs.getDipoleMatrixElement(**transition)
arc_radial = arc_cs.getRadialMatrixElement(
    transition["n1"], transition["l1"], transition["j1"],
    transition["n2"], transition["l2"], transition["j2"],
)
arc_reduced_j = arc_cs.getReducedMatrixElementJ(
    transition["n1"], transition["l1"], transition["j1"],
    transition["n2"], transition["l2"], transition["j2"],
)
arc_wavelength_m = arc_cs.getTransitionWavelength(
    transition["n1"], transition["l1"], transition["j1"],
    transition["n2"], transition["l2"], transition["j2"],
)
arc_frequency_hz = cs.c / arc_wavelength_m
arc_8p_binding_cm = arc_cs.getEnergy(8, 1, 0.5) * cm_per_ev
arc_52d_binding_cm = arc_cs.getEnergy(52, 2, 1.5) * cm_per_ev
arc_8p_level_cm = cs_ionization_cm + arc_8p_binding_cm
arc_52d_level_cm = cs_ionization_cm + arc_52d_binding_cm

print_power_block(
    "ARC / Caesium / pi",
    arc_dme,
    arc_wavelength_m,
    {
        "8P1/2 level above ground [cm^-1]": f"{arc_8p_level_cm:.6f}",
        "52D3/2 level above ground [cm^-1]": f"{arc_52d_level_cm:.6f}",
        "radial <nlj|r|n'l'j'> [a0]": f"{arc_radial:+.12e}",
        "reduced-J DME [e a0]": f"{arc_reduced_j:+.12e}",
        "transition frequency [THz]": f"{arc_frequency_hz / 1e12:.6f}",
    },
)

ARC / Caesium / pi
  lambda = 1769.006451 nm
  d_eff = +3.808585780484e-02 e a0 = +3.229053703824e-31 C m
  8P1/2 level above ground [cm^-1] = 25708.835393
  52D3/2 level above ground [cm^-1] = 31361.726077
  radial <nlj|r|n'l'j'> [a0] = +8.079230496332e-02
  reduced-J DME [e a0] = -9.329091803805e-02
  transition frequency [THz] = 169.469398
  Gaussian waist: wx = wy = 10.0 um
  Omega/2pi =  1.0 MHz -> P = 8.778510588e-07 W = 0.000878 mW, I0 = 5.588573e+03 W/m^2, E0 = 2.052016e+03 V/m
  Omega/2pi =  5.0 MHz -> P = 2.194627647e-05 W = 0.021946 mW, I0 = 1.397143e+05 W/m^2, E0 = 1.026008e+04 V/m
  Omega/2pi = 10.0 MHz -> P = 8.778510588e-05 W = 0.087785 mW, I0 = 5.588573e+05 W/m^2, E0 = 2.052016e+04 V/m


In [4]:
ryd_cs = rydcalc.Cesium133(use_db=False)
ryd_initial = ryd_cs.get_state((transition["n1"], transition["l1"], transition["j1"], transition["mj1"]))
ryd_final = ryd_cs.get_state((transition["n2"], transition["l2"], transition["j2"], transition["mj2"]))

ryd_dme = ryd_initial.get_multipole_me(ryd_final, k=1, qIn=[transition["q"]])
ryd_frequency_hz = abs(ryd_final.energy_Hz - ryd_initial.energy_Hz)
ryd_wavelength_m = cs.c / ryd_frequency_hz
ryd_8p_binding_cm = ryd_initial.energy_Hz / (100.0 * cs.c)
ryd_52d_binding_cm = ryd_final.energy_Hz / (100.0 * cs.c)
ryd_8p_level_cm = cs_ionization_cm + ryd_8p_binding_cm
ryd_52d_level_cm = cs_ionization_cm + ryd_52d_binding_cm

print_power_block(
    "rydcalc / Cesium133 / pi",
    ryd_dme,
    ryd_wavelength_m,
    {
        "initial state": ryd_cs.repr_state(ryd_initial),
        "final state": ryd_cs.repr_state(ryd_final),
        "8P1/2 level above ground [cm^-1]": f"{ryd_8p_level_cm:.6f}",
        "52D3/2 level above ground [cm^-1]": f"{ryd_52d_level_cm:.6f}",
        "transition frequency [THz]": f"{ryd_frequency_hz / 1e12:.6f}",
    },
)

rydcalc / Cesium133 / pi
  lambda = 1769.914537 nm
  d_eff = +3.815763233291e-02 e a0 = +3.235139002123e-31 C m
  initial state = |Cs:8,P,0.5,0.5>
  final state = |Cs:52,D,1.5,0.5>
  8P1/2 level above ground [cm^-1] = 25711.735519
  52D3/2 level above ground [cm^-1] = 31361.725839
  transition frequency [THz] = 169.382449
  Gaussian waist: wx = wy = 10.0 um
  Omega/2pi =  1.0 MHz -> P = 8.745516881e-07 W = 0.000875 mW, I0 = 5.567569e+03 W/m^2, E0 = 2.048156e+03 V/m
  Omega/2pi =  5.0 MHz -> P = 2.186379220e-05 W = 0.021864 mW, I0 = 1.391892e+05 W/m^2, E0 = 1.024078e+04 V/m
  Omega/2pi = 10.0 MHz -> P = 8.745516881e-05 W = 0.087455 mW, I0 = 5.567569e+05 W/m^2, E0 = 2.048156e+04 V/m


In [5]:
arc_power_1mhz = gaussian_power_for_rabi(1e6, arc_dme, w_x, w_y)["power_w"]
ryd_power_1mhz = gaussian_power_for_rabi(1e6, ryd_dme, w_x, w_y)["power_w"]

rows = [
    ("lambda [nm]", 1e9 * arc_wavelength_m, 1e9 * ryd_wavelength_m),
    ("frequency [THz]", arc_frequency_hz / 1e12, ryd_frequency_hz / 1e12),
    ("8P level [cm^-1]", arc_8p_level_cm, ryd_8p_level_cm),
    ("52D level [cm^-1]", arc_52d_level_cm, ryd_52d_level_cm),
    ("d_eff [e a0]", arc_dme, ryd_dme),
    ("|d_eff| [e a0]", abs(arc_dme), abs(ryd_dme)),
    ("P at 1 MHz [mW]", 1e3 * arc_power_1mhz, 1e3 * ryd_power_1mhz),
]

print("ARC vs rydcalc direct comparison")
print(f"{'quantity':<22} {'ARC':>18} {'rydcalc':>18} {'rydcalc/ARC':>14} {'rel diff':>12}")
for name, arc_value, ryd_value in rows:
    ratio = ryd_value / arc_value if arc_value != 0 else math.nan
    rel_diff = (ryd_value - arc_value) / arc_value if arc_value != 0 else math.nan
    print(f"{name:<22} {arc_value:>18.9e} {ryd_value:>18.9e} {ratio:>14.6f} {rel_diff:>12.3e}")

print()
print("Diagnostic conclusion:")
print("  ARC and rydcalc agree closely for this 8P1/2 -> 52D3/2 pi transition.")
print("  Wavelength differs by {:.3f} nm ({:.3e} relative).".format(
    1e9 * (ryd_wavelength_m - arc_wavelength_m),
    (ryd_wavelength_m - arc_wavelength_m) / arc_wavelength_m,
))
print("  |d_eff| differs by {:.3e} relative.".format((abs(ryd_dme) - abs(arc_dme)) / abs(arc_dme)))
print("  Power differs by {:.3e} relative because P scales as 1/|d_eff|^2.".format((ryd_power_1mhz - arc_power_1mhz) / arc_power_1mhz))

ARC vs rydcalc direct comparison
quantity                              ARC            rydcalc    rydcalc/ARC     rel diff
lambda [nm]               1.769006451e+03    1.769914537e+03       1.000513    5.133e-04
frequency [THz]           1.694693979e+02    1.693824486e+02       0.999487   -5.131e-04
8P level [cm^-1]          2.570883539e+04    2.571173552e+04       1.000113    1.128e-04
52D level [cm^-1]         3.136172608e+04    3.136172584e+04       1.000000   -7.565e-09
d_eff [e a0]              3.808585780e-02    3.815763233e-02       1.001885    1.885e-03
|d_eff| [e a0]            3.808585780e-02    3.815763233e-02       1.001885    1.885e-03
P at 1 MHz [mW]           8.778510588e-04    8.745516881e-04       0.996242   -3.758e-03

Diagnostic conclusion:
  ARC and rydcalc agree closely for this 8P1/2 -> 52D3/2 pi transition.
  Wavelength differs by 0.908 nm (5.133e-04 relative).
  |d_eff| differs by 1.885e-03 relative.
  Power differs by -3.758e-03 relative because P scales as 1/|d

## Literature check: which result is more credible?

This transition is very different from the previous `6S -> 60P` validation case. Here both packages produce a physically consistent result:

- Both give a near-infrared transition around `1.769 um`.
- Both put `52D3/2` about `44.74 cm^-1` below the Cs ionization limit.
- Both give nearly the same `pi` effective DME, about `0.0381 e a0`.

The literature/database check supports this scale:

- NIST ASD / NIST Cs data provide critically evaluated Cs energy levels and the Cs ionization limit. The ARC packaged Cs data include the NIST `8P1/2` energy, so ARC has a measured low-lying-state anchor for the lower state.
- Shen et al., Phys. Rev. Lett. 133, 233005 (2024), report high-precision Cs `nD3/2,5/2` quantum defects for `n=21-90`, directly covering `52D3/2`. This is the right class of reference for the Rydberg upper state.
- Bhowmik et al., Phys. Rev. A 110, 043114 (2024), discuss Cs `nD_{J,|M_J|}` Rydberg states and explicitly identify the `nD_{3/2}, |M_J|=1/2` series with auxiliary `8P1/2` states for magic-wavelength trapping in the `1000-2000 nm` range. This matches the role of the `8P1/2 <-> 52D3/2` transition here.
- Bao et al., Scientific Reports 14, 7779 (2024), measure relative transition strengths of Cs Rydberg D states for `n=40-62` using Rydberg EIT and find agreement between experiment and theory for D-state transition-strength ratios. This supports the angular-coupling/radial-wavefunction scale for the same Rydberg-D range, though not this exact `8P` lower state.

Credibility judgment:

- Use ARC as the primary quoted value because its low-lying `8P1/2` state is tied to NIST data and its API is the established alkali reference for optical matrix elements.
- In contrast to the previous `6S -> 60P` case, local `rydcalc.Cesium133` passes this validation: its `8P -> 52D` wavelength and DME agree with ARC at the sub-percent level.
- For practical estimates, the ARC and rydcalc powers are interchangeable at the precision implied by the simple Gaussian-beam model. The remaining difference is much smaller than typical experimental uncertainties from waist, polarization purity, and delivered-power calibration.

Reference links:

- NIST ASD contents and scope: https://www.nist.gov/pml/data/asd_contents.cfm
- NIST Cs atomic data / ionization limit: https://physics.nist.gov/PhysRefData/Handbook/Tables/cesiumtable1_a.htm
- NIST Cs neutral levels summary: https://physics.nist.gov/PhysRefData/Handbook/Tables/cesiumtable5.htm
- Shen et al., `Ultraprecise Determination of Cs (nS1/2) and Cs (nDJ) Quantum Defects for Sensing and Computing`, Phys. Rev. Lett. 133, 233005 (2024): https://doi.org/10.1103/PhysRevLett.133.233005
- Bhowmik et al., `Double, triple, and quadruple magic wavelengths for cesium ground, excited, and Rydberg states`, Phys. Rev. A 110, 043114 (2024): https://doi.org/10.1103/PhysRevA.110.043114
- Bao et al., `Measurement of relative transition strengths of 133Cs Rydberg D states using electromagnetically induced transparency`, Scientific Reports 14, 7779 (2024): https://doi.org/10.1038/s41598-024-58385-0
- ARC transition-wavelength API documentation: https://arc-alkali-rydberg-calculator.readthedocs.io/en/latest/generated/arc.alkali_atom_functions.AlkaliAtom.getTransitionWavelength.html


## Adopted estimate

For the requested `133Cs |8P1/2, mj=1/2> -> |52D3/2, mj=1/2>` `pi` transition, adopt the ARC value as the main benchmark:

```text
lambda = 1769.006451 nm
|d_eff| = 3.808585780484e-02 e a0
w_x = w_y = 10 um
Omega / (2 pi) = 10 MHz
P = 0.087785 mW
```

The rydcalc value is a successful cross-check:

```text
lambda = 1769.914537 nm
|d_eff| = 3.815763233291e-02 e a0
Omega / (2 pi) = 10 MHz
P = 0.087455 mW
```

This is a passed validation case for local rydcalc on an excited low-lying `P` state coupled to a high-`n` Cs `D` Rydberg state. For experimental planning, uncertainty in beam waist dominates this ARC-vs-rydcalc difference because `P ∝ w_x w_y / |d_eff|^2`.
